In [1]:
# ============================================================
# Cell 1 -- Imports, Configuration & Reproducibility
# ============================================================

import os
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# -- Reproducibility -----------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# -- CPU thread cap ------------------------------------------
# Limits the OpenMP/MKL thread pool PyTorch uses for matrix
# mults and convolutions. Without this, PyTorch claims every
# logical core. Must be set before any tensor ops.
torch.set_num_threads(2)
torch.set_num_interop_threads(1)

# -- Device --------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch intra-op threads : {torch.get_num_threads()}")
print(f"PyTorch inter-op threads : {torch.get_num_interop_threads()}")

# -- Sequence / Feature Constants ----------------------------
SEQUENCE_LENGTH       = 90
MAX_SEQUENCE_LENGTH   = 120   # 90 real frames + 30 zero buffer
N_FEATURES            = 26   # pose feature columns (angles + velocities)
N_EXERCISE_CLASSES    = 4    # pullup=0, pushup=1, russian_twist=2, squat=3
N_CORRECTNESS_CLASSES = 2    # incorrect=0, correct=1

# -- Training Constants --------------------------------------
BATCH_SIZE = 32
N_EPOCHS   = 50              # safe ceiling; early stopping fires first
LR         = 1e-3
WEIGHT_DECAY = 1e-4

# -- Columns that are NOT model input features ---------------
NON_FEATURE_COLS = [
    "video_id",
    "frame_number",
    "exercise_name_encoded",
    "exercise_correctness_encoded",
]

# -- Paths ---------------------------------------------------
DATA_DIR  = Path("../data/processed/tabular_data")
MODEL_DIR = Path("../data/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = Path("../data/figures/07_deep_learning")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATA_DIR / "train.csv"
VAL_CSV   = DATA_DIR / "val.csv"
TEST_CSV  = DATA_DIR / "test.csv"

# -- Sanity check: verify CSVs exist -------------------------
for p in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    assert p.exists(), f"Missing file: {p}"
    print(f"  Found: {p}")

print("Cell 1 complete -- config & reproducibility ready.")

Device: cpu
PyTorch intra-op threads : 2
PyTorch inter-op threads : 1
  Found: ..\data\processed\tabular_data\train.csv
  Found: ..\data\processed\tabular_data\val.csv
  Found: ..\data\processed\tabular_data\test.csv
Cell 1 complete -- config & reproducibility ready.


In [2]:
class SequenceDataset(Dataset):
    """
    Builds non-overlapping windows from pre-split CSVs.

    Window size per video: min(n_frames, sequence_length).
    Videos shorter than sequence_length are used as a single window
    and zero-padded to sequence_length so all tensors share the same shape.

    Per-video grouping guarantees no cross-video contamination.
    Labels assigned via mode across the window (guards against
    stray mislabeled rows surviving the cleaning step).
    All tensors are pre-built in RAM at load time to avoid
    repeated disk I/O during training.
    """

    def __init__(self, csv_path: Path,
                 sequence_length: int = SEQUENCE_LENGTH,
                 max_len: int = MAX_SEQUENCE_LENGTH,
                 augment: bool = False):
        self.augment         = augment
        self.sequence_length = sequence_length
        self.max_len         = max_len
        df = pd.read_csv(csv_path)

        # ── Identify feature columns ─────────────────────────
        feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]

        # Hard assertion: catch column mismatches immediately
        assert len(feature_cols) == N_FEATURES, (
            f"Expected {N_FEATURES} feature columns, found {len(feature_cols)}."
            f"Columns found: {feature_cols}"
        )

        sequences, labels_ex, labels_cor, win_sizes = [], [], [], []

        # ── Build windows per video ──────────────────────────
        for video_id, group in df.groupby("video_id"):
            group = group.sort_values("frame_number").reset_index(drop=True)

            features = group[feature_cols].values          # (n_frames, 26)
            ex_labels = group["exercise_name_encoded"].values
            cor_labels = group["exercise_correctness_encoded"].values

            n_frames = len(group)
            # Short clips (<=sequence_length): one window using every frame.
            # Long clips: non-overlapping sequence_length-frame chunks.
            win_size = n_frames if n_frames <= sequence_length else sequence_length

            for start in range(0, n_frames - win_size + 1, win_size):
                window  = features[start : start + win_size]
                ex_win  = ex_labels[start  : start + win_size]
                cor_win = cor_labels[start : start + win_size]

                # Zero-pad to max_len so all tensors share the same shape.
                # Convention: real frames first, zeros at the end.
                if win_size < max_len:
                    pad = max_len - win_size
                    window  = np.vstack([window, np.zeros((pad, window.shape[1]), dtype=window.dtype)])
                    ex_win  = np.pad(ex_win,  (0, pad), mode='edge')
                    cor_win = np.pad(cor_win, (0, pad), mode='edge')

                # Label = mode across window frames
                ex_label  = int(np.bincount(ex_win).argmax())
                cor_label = int(np.bincount(cor_win).argmax())

                sequences.append(window)
                labels_ex.append(ex_label)
                labels_cor.append(cor_label)
                win_sizes.append(win_size)

            # Keep trailing partial chunk if it is longer than 20 frames.
            # Short clips have win_size == n_frames, so remainder is always 0.
            remainder = n_frames % win_size
            if remainder > 20:
                start   = n_frames - remainder
                window  = features[start : start + remainder]
                ex_win  = ex_labels[start  : start + remainder]
                cor_win = cor_labels[start : start + remainder]

                pad = max_len - remainder
                window  = np.vstack([window, np.zeros((pad, window.shape[1]), dtype=window.dtype)])
                ex_win  = np.pad(ex_win,  (0, pad), mode='edge')
                cor_win = np.pad(cor_win, (0, pad), mode='edge')

                ex_label  = int(np.bincount(ex_win).argmax())
                cor_label = int(np.bincount(cor_win).argmax())

                sequences.append(window)
                labels_ex.append(ex_label)
                labels_cor.append(cor_label)
                win_sizes.append(remainder)

        # ── Pre-build tensors in RAM ─────────────────────────
        self.X   = torch.tensor(np.array(sequences), dtype=torch.float32)
        self.y_ex  = torch.tensor(labels_ex,  dtype=torch.long)
        self.y_cor = torch.tensor(labels_cor, dtype=torch.long)
        self.win_sizes = win_sizes

        print(f"  Loaded {csv_path.name}: "
              f"{len(self.X)} sequences "
              f"from {df['video_id'].nunique()} videos")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x        = self.X[idx]
        win_size = self.win_sizes[idx]
        pad      = self.max_len - win_size

        if self.augment:
            # Training only: randomly place the zero block before or after real
            # frames (50/50) so the model learns position-invariant recognition.
            if torch.rand(1).item() < 0.5:
                x_placed       = torch.zeros_like(x)
                x_placed[pad:] = x[:win_size]
                x = x_placed
            x = x + torch.randn_like(x) * 0.05

        # Val / test (augment=False): tensors are returned as-is — real frames
        # first, zeros at the end — giving deterministic, reproducible evaluation.
        return x, self.y_ex[idx], self.y_cor[idx]


# ── Instantiate datasets ─────────────────────────────────────
print("Building datasets...")
train_dataset = SequenceDataset(TRAIN_CSV, augment=True)  # noise aug — val/test are never augmented
val_dataset   = SequenceDataset(VAL_CSV)
test_dataset  = SequenceDataset(TEST_CSV)

# ── Wrap in DataLoaders ──────────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,           # shuffle sequences (not frames) each epoch
    num_workers=0,          # no subprocess overhead on laptop
    pin_memory=False,       # only beneficial with CUDA
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

# ── Summary ──────────────────────────────────────────────────
print(f"Sequence tensor shapes:")
print(f"  Train X : {train_dataset.X.shape}   "
      f"y_exercise: {train_dataset.y_ex.shape}   "
      f"y_correctness: {train_dataset.y_cor.shape}")
print(f"  Val   X : {val_dataset.X.shape}")
print(f"  Test  X : {test_dataset.X.shape}")

print(f"Exercise class distribution (train):")
for cls, name in enumerate(["pullup", "pushup", "russian_twist", "squat"]):
    count = (train_dataset.y_ex == cls).sum().item()
    print(f"  {cls} — {name:<15}: {count} sequences")

print(f"Correctness class distribution (train):")
for cls, name in enumerate(["incorrect", "correct"]):
    count = (train_dataset.y_cor == cls).sum().item()
    print(f"  {cls} — {name:<10}: {count} sequences")

print("Cell 2 complete — datasets and loaders ready.")

Building datasets...
  Loaded train.csv: 250 sequences from 170 videos
  Loaded val.csv: 57 sequences from 37 videos
  Loaded test.csv: 56 sequences from 37 videos
Sequence tensor shapes:
  Train X : torch.Size([250, 120, 26])   y_exercise: torch.Size([250])   y_correctness: torch.Size([250])
  Val   X : torch.Size([57, 120, 26])
  Test  X : torch.Size([56, 120, 26])
Exercise class distribution (train):
  0 — pullup         : 69 sequences
  1 — pushup         : 56 sequences
  2 — russian_twist  : 58 sequences
  3 — squat          : 67 sequences
Correctness class distribution (train):
  0 — incorrect : 133 sequences
  1 — correct   : 117 sequences
Cell 2 complete — datasets and loaders ready.


In [3]:
# ============================================================
# Cell 3 — Model Definitions (ExerciseLSTM & ExerciseTCN)
# ============================================================

# ── ExerciseLSTM ─────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        # Adding a tanh layer makes the attention "scoring" more expressive
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, encoder_outputs):
        # encoder_outputs: [batch, seq_len, hidden_dim]
        weights = self.attention(encoder_outputs).squeeze(-1) # [batch, seq_len]
        weights = F.softmax(weights, dim=1)
        # weighted sum of hidden states
        context = torch.sum(weights.unsqueeze(-1) * encoder_outputs, dim=1) # [batch, hidden_dim]
        return context

class ExerciseLSTM(nn.Module):
    def __init__(self, input_size, n_exercise, n_correctness, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        # Attention over bidirectional outputs (hidden_size * 2)
        self.attention = Attention(hidden_size * 2)  # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        
        self.fc_shared = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),  # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.head_exercise = nn.Linear(64, n_exercise)
        # Soft MoE: one correctness head per exercise, mixed by exercise probabilities.
        self.heads_correctness = nn.ModuleList(
            [nn.Linear(64, n_correctness) for _ in range(n_exercise)]
        )

    def forward(self, x):
        # x: [batch, seq_len, features]
        lstm_out, _ = self.lstm(x) # [batch, seq_len, hidden*2]
        
        # Apply Attention to get context vector
        context = self.attention(lstm_out) # [batch, hidden*2]
        
        shared    = self.fc_shared(context)
        ex_logits = self.head_exercise(shared)
        mix       = F.softmax(ex_logits, dim=1)          # [batch, n_exercise]
        cor_logits = sum(
            mix[:, i].unsqueeze(1) * head(shared)
            for i, head in enumerate(self.heads_correctness)
        )                                                 # [batch, n_correctness]
        return ex_logits, cor_logits



# ── TCN Building Blocks ───────────────────────────────────────

class Chomp1d(nn.Module):
    """Removes the right-side padding to enforce causality."""
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        # x: [batch, channels, seq_len + chomp_size]
        return x[:, :, : -self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    """
    One dilated causal convolutional block with residual connection.

    Conv1 -> Chomp -> ReLU -> Dropout
    Conv2 -> Chomp -> ReLU -> Dropout
    + residual (1x1 conv projection if channel dims differ)
    -> ReLU
    """

    def __init__(
        self,
        in_channels:  int,
        out_channels: int,
        kernel_size:  int,
        dilation:     int,
        dropout:      float = 0.2,
    ):
        super().__init__()
        # Left-only padding = (kernel_size - 1) * dilation
        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.utils.weight_norm(
            nn.Conv1d(in_channels, out_channels, kernel_size,
                      padding=padding, dilation=dilation)
        )
        self.chomp1  = Chomp1d(padding)
        self.relu1   = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.utils.weight_norm(
            nn.Conv1d(out_channels, out_channels, kernel_size,
                      padding=padding, dilation=dilation)
        )
        self.chomp2   = Chomp1d(padding)
        self.relu2    = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.relu1, self.dropout1,
            self.conv2, self.chomp2, self.relu2, self.dropout2,
        )

        # Residual projection: only needed when channel dims differ
        self.residual_proj = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else None
        )

        self.final_relu = nn.ReLU()
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.conv1.weight, mean=0.0, std=0.01)
        nn.init.normal_(self.conv2.weight, mean=0.0, std=0.01)

    def forward(self, x):
        out = self.net(x)
        res = x if self.residual_proj is None else self.residual_proj(x)
        return self.final_relu(out + res)


# ── ExerciseTCN ──────────────────────────────────────────────

class ExerciseTCN(nn.Module):
    """
    Temporal Convolutional Network with dual output heads.

    Dilations [1, 2, 4] give a receptive field of:
        1 + (kernel_size-1) * (1+2+4) * 2 = 1 + 2*7*2 = 29 frames
    which covers the full 30-frame window.

    Architecture:
        Input   : [batch, SEQUENCE_LENGTH, N_FEATURES]
        Permute : [batch, N_FEATURES, SEQUENCE_LENGTH]   (Conv1d format)
        Block 1 : dilation=1, in=26,  out=32
        Block 2 : dilation=2, in=32,  out=64
        Block 3 : dilation=4, in=64,  out=64
        Trunk   : Linear(64, 64) -> ReLU -> Dropout
        Head 1  : Linear(64, N_EXERCISE_CLASSES)
        Head 2  : Soft MoE -- N_EXERCISE_CLASSES x Linear(64, N_CORRECTNESS_CLASSES)
    """

    def __init__(
        self,
        input_size:   int   = N_FEATURES,
        channels:     list  = None,
        kernel_size:  int   = 3,
        dilations:    list  = None,
        dropout:      float = 0.2,
    ):
        super().__init__()
        channels  = channels  or [32, 64, 64]
        dilations = dilations or [1, 2, 4]

        assert len(channels) == len(dilations), (
            "channels and dilations must have the same length."
        )

        # Stack of TemporalBlocks
        blocks = []
        in_ch  = input_size
        for out_ch, dil in zip(channels, dilations):
            blocks.append(TemporalBlock(in_ch, out_ch, kernel_size, dil, dropout))
            in_ch = out_ch
        self.network = nn.Sequential(*blocks)

        self.fc_shared = nn.Sequential(
            nn.Linear(channels[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.head_exercise    = nn.Linear(64, N_EXERCISE_CLASSES)
        # Soft MoE: one correctness head per exercise, mixed by exercise probabilities.
        self.heads_correctness = nn.ModuleList(
            [nn.Linear(64, N_CORRECTNESS_CLASSES) for _ in range(N_EXERCISE_CLASSES)]
        )

    def forward(self, x):
        # x: [batch, seq_len, features]
        # Detect real frames before transposing: padding rows are all-zero.
        mask = (x.abs().sum(dim=2) > 0).float()          # [batch, seq_len]

        x   = x.transpose(1, 2)                          # [batch, features, seq_len]
        out = self.network(x)                             # [batch, channels[-1], seq_len]

        # Masked average pool over real frames only — avoids diluting the
        # representation with zero-padded tail positions.
        mask_3d  = mask.unsqueeze(1)                      # [batch, 1, seq_len]
        out_pool = (out * mask_3d).sum(dim=2) / mask.sum(dim=1, keepdim=True).clamp(min=1)

        shared    = self.fc_shared(out_pool)              # [batch, 64]
        ex_logits = self.head_exercise(shared)
        mix       = F.softmax(ex_logits, dim=1)
        cor_logits = sum(
            mix[:, i].unsqueeze(1) * head(shared)
            for i, head in enumerate(self.heads_correctness)
        )
        return ex_logits, cor_logits


# ── Sanity Check ─────────────────────────────────────────────

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

dummy = torch.randn(2, MAX_SEQUENCE_LENGTH, N_FEATURES)  # batch of 2

print("── ExerciseLSTM ──────────────────────────────")
# Ensure you pass your class counts here
lstm_model = ExerciseLSTM(
    input_size=N_FEATURES, 
    n_exercise=N_EXERCISE_CLASSES, 
    n_correctness=N_CORRECTNESS_CLASSES
)

ex_out, cor_out = lstm_model(dummy)

print(f"  Input shape        : {list(dummy.shape)}")
print(f"  Exercise logits    : {list(ex_out.shape)}   (expected [batch, {N_EXERCISE_CLASSES}])")
print(f"  Correctness logits : {list(cor_out.shape)}  (expected [batch, {N_CORRECTNESS_CLASSES}])")
print(f"  Trainable params   : {sum(p.numel() for p in lstm_model.parameters() if p.requires_grad):,}")

# Gradient check: all 4 correctness heads must receive gradient
(ex_out.sum() + cor_out.sum()).backward()
for i, h in enumerate(lstm_model.heads_correctness):
    assert h.weight.grad is not None, f"No grad for heads_correctness[{i}]"
print("  Gradient check     : all heads_correctness receive gradient")

print()
print("── ExerciseTCN ───────────────────────────────")
tcn_model = ExerciseTCN()
ex_out, cor_out = tcn_model(dummy)
print(f"  Input shape        : {list(dummy.shape)}")
print(f"  Exercise logits    : {list(ex_out.shape)}   (expected [2, {N_EXERCISE_CLASSES}])")
print(f"  Correctness logits : {list(cor_out.shape)}  (expected [2, {N_CORRECTNESS_CLASSES}])")
print(f"  Trainable params   : {count_parameters(tcn_model):,}")

(ex_out.sum() + cor_out.sum()).backward()
for i, h in enumerate(tcn_model.heads_correctness):
    assert h.weight.grad is not None, f"No grad for heads_correctness[{i}]"
print("  Gradient check     : all heads_correctness receive gradient")

print("\nCell 3 complete -- both models defined and verified.")


── ExerciseLSTM ──────────────────────────────
  Input shape        : [2, 120, 26]
  Exercise logits    : [2, 4]   (expected [batch, 4])
  Correctness logits : [2, 2]  (expected [batch, 2])
  Trainable params   : 163,789
  Gradient check     : all heads_correctness receive gradient

── ExerciseTCN ───────────────────────────────
  Input shape        : [2, 120, 26]
  Exercise logits    : [2, 4]   (expected [2, 4])
  Correctness logits : [2, 2]  (expected [2, 2])
  Trainable params   : 57,132
  Gradient check     : all heads_correctness receive gradient

Cell 3 complete -- both models defined and verified.


c:\Users\USER\Desktop\AI_PROJECT_2\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [4]:
# ============================================================
# Cell 4 - Optimized Training Loop with Hyperparameter Tuning
# ============================================================

import copy
import shutil
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

# ============================================================
# 1. Reproducibility
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 2. Split leakage check
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

train_video_ids = set(train_df["video_id"].unique())
val_video_ids = set(val_df["video_id"].unique())
test_video_ids = set(test_df["video_id"].unique())

assert train_video_ids.isdisjoint(val_video_ids), "Leakage: train and val share videos."
assert train_video_ids.isdisjoint(test_video_ids), "Leakage: train and test share videos."
assert val_video_ids.isdisjoint(test_video_ids), "Leakage: val and test share videos."

feature_cols = [c for c in train_df.columns if c not in NON_FEATURE_COLS]

print("Video split check passed.")
print(f"Train videos: {len(train_video_ids)}")
print(f"Val videos  : {len(val_video_ids)}")
print(f"Test videos : {len(test_video_ids)}")


# ============================================================
# 3. Train-only feature scaling
# ============================================================

for dataset in [train_dataset, val_dataset, test_dataset]:
    if not hasattr(dataset, "X_raw"):
        dataset.X_raw = dataset.X.clone()

train_dataset.X = train_dataset.X_raw.clone()
val_dataset.X = val_dataset.X_raw.clone()
test_dataset.X = test_dataset.X_raw.clone()

train_mean = train_dataset.X.reshape(-1, N_FEATURES).mean(dim=0)
train_std = train_dataset.X.reshape(-1, N_FEATURES).std(dim=0)
train_std[train_std == 0] = 1.0

train_dataset.X = (train_dataset.X - train_mean) / train_std
val_dataset.X = (val_dataset.X - train_mean) / train_std
test_dataset.X = (test_dataset.X - train_mean) / train_std

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

print("Train-only feature scaling applied.")


# ============================================================
# 4. Class weights
# ============================================================

def make_balanced_weights(labels, n_classes):
    labels = labels.cpu().numpy()
    counts = np.bincount(labels, minlength=n_classes)
    counts = np.maximum(counts, 1)

    total = counts.sum()
    weights = total / (n_classes * counts)

    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


exercise_class_weights = make_balanced_weights(
    train_dataset.y_ex,
    N_EXERCISE_CLASSES
)

correctness_class_weights = make_balanced_weights(
    train_dataset.y_cor,
    N_CORRECTNESS_CLASSES
)

print("Exercise class weights   :", exercise_class_weights.detach().cpu().numpy())
print("Correctness class weights:", correctness_class_weights.detach().cpu().numpy())


# ============================================================
# 5. Video-level validation helpers
# ============================================================

def build_window_video_ids(csv_path, sequence_length, max_len=MAX_SEQUENCE_LENGTH):
    df = pd.read_csv(csv_path)
    window_video_ids = []

    for video_id, group in df.groupby("video_id"):
        group = group.sort_values("frame_number").reset_index(drop=True)
        n_frames = len(group)

        win_size = n_frames if n_frames <= sequence_length else sequence_length

        for start in range(0, n_frames - win_size + 1, win_size):
            window_video_ids.append(video_id)

        remainder = n_frames % win_size
        if remainder > 20:
            window_video_ids.append(video_id)

    return np.array(window_video_ids)


VAL_WINDOW_VIDEO_IDS = build_window_video_ids(VAL_CSV, SEQUENCE_LENGTH)

if len(VAL_WINDOW_VIDEO_IDS) != len(val_dataset):
    print("Warning: video-level validation disabled because window IDs do not match dataset length.")
    VAL_WINDOW_VIDEO_IDS = None
else:
    print("Video-level validation enabled.")


def majority_label(labels):
    labels = np.asarray(labels, dtype=int)
    return int(np.bincount(labels).argmax())


def aggregate_by_video(video_ids, ex_probs, cor_probs, ex_labels, cor_labels):
    video_ids = np.asarray(video_ids)

    video_ex_preds = []
    video_ex_labels = []
    video_cor_preds = []
    video_cor_labels = []

    for video_id in np.unique(video_ids):
        idx = np.where(video_ids == video_id)[0]

        ex_pred = int(ex_probs[idx].mean(axis=0).argmax())
        cor_pred = int(cor_probs[idx].mean(axis=0).argmax())

        ex_label = majority_label(ex_labels[idx])
        cor_label = majority_label(cor_labels[idx])

        video_ex_preds.append(ex_pred)
        video_ex_labels.append(ex_label)
        video_cor_preds.append(cor_pred)
        video_cor_labels.append(cor_label)

    return {
        "video_f1_ex_macro": f1_score(
            video_ex_labels,
            video_ex_preds,
            average="macro",
            zero_division=0
        ),
        "video_f1_cor_macro": f1_score(
            video_cor_labels,
            video_cor_preds,
            average="macro",
            zero_division=0
        ),
        "video_f1_ex_weighted": f1_score(
            video_ex_labels,
            video_ex_preds,
            average="weighted",
            zero_division=0
        ),
        "video_f1_cor_weighted": f1_score(
            video_cor_labels,
            video_cor_preds,
            average="weighted",
            zero_division=0
        ),
    }


def evaluate_model(model, loader, video_ids=None):
    model.eval()

    all_ex_probs = []
    all_cor_probs = []
    all_ex_preds = []
    all_cor_preds = []
    all_ex_labels = []
    all_cor_labels = []

    with torch.no_grad():
        for X_batch, y_ex_batch, y_cor_batch in loader:
            X_batch = X_batch.to(DEVICE)

            logits_ex, logits_cor = model(X_batch)

            ex_probs = torch.softmax(logits_ex, dim=1).cpu().numpy()
            cor_probs = torch.softmax(logits_cor, dim=1).cpu().numpy()

            ex_preds = ex_probs.argmax(axis=1)
            cor_preds = cor_probs.argmax(axis=1)

            all_ex_probs.append(ex_probs)
            all_cor_probs.append(cor_probs)

            all_ex_preds.extend(ex_preds)
            all_cor_preds.extend(cor_preds)
            all_ex_labels.extend(y_ex_batch.numpy())
            all_cor_labels.extend(y_cor_batch.numpy())

    all_ex_probs = np.vstack(all_ex_probs)
    all_cor_probs = np.vstack(all_cor_probs)
    all_ex_preds = np.array(all_ex_preds)
    all_cor_preds = np.array(all_cor_preds)
    all_ex_labels = np.array(all_ex_labels)
    all_cor_labels = np.array(all_cor_labels)

    metrics = {
        "window_f1_ex_macro": f1_score(
            all_ex_labels,
            all_ex_preds,
            average="macro",
            zero_division=0
        ),
        "window_f1_cor_macro": f1_score(
            all_cor_labels,
            all_cor_preds,
            average="macro",
            zero_division=0
        ),
        "window_f1_ex_weighted": f1_score(
            all_ex_labels,
            all_ex_preds,
            average="weighted",
            zero_division=0
        ),
        "window_f1_cor_weighted": f1_score(
            all_cor_labels,
            all_cor_preds,
            average="weighted",
            zero_division=0
        ),
    }

    if video_ids is not None and len(video_ids) == len(all_ex_labels):
        video_metrics = aggregate_by_video(
            video_ids,
            all_ex_probs,
            all_cor_probs,
            all_ex_labels,
            all_cor_labels
        )
        metrics.update(video_metrics)

        metrics["score"] = (
            0.15 * metrics["video_f1_ex_macro"]
            + 0.85 * metrics["video_f1_cor_macro"]
        )
    else:
        metrics["score"] = (
            0.15 * metrics["window_f1_ex_macro"]
            + 0.85 * metrics["window_f1_cor_macro"]
        )

    return metrics


# ============================================================
# 6. Model builders
# ============================================================

def build_lstm(cfg):
    try:
        return ExerciseLSTM(
            input_size=N_FEATURES,
            n_exercise=N_EXERCISE_CLASSES,
            n_correctness=N_CORRECTNESS_CLASSES,
            hidden_size=cfg["hidden_size"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"],
        )
    except TypeError:
        return ExerciseLSTM(
            input_size=N_FEATURES,
            hidden_size=cfg["hidden_size"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"],
        )


def build_tcn(cfg):
    try:
        return ExerciseTCN(
            input_size=N_FEATURES,
            channels=cfg["channels"],
            kernel_size=cfg.get("kernel_size", 3),
            dilations=cfg.get("dilations", None),
            dropout=cfg["dropout"],
        )
    except TypeError:
        return ExerciseTCN(
            channels=cfg["channels"],
            dropout=cfg["dropout"],
        )


# ============================================================
# 7. Stronger tuning grids
# ============================================================

LSTM_GRID = [
    {
        "hidden_size": 32,
        "num_layers": 1,
        "dropout": 0.20,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "correctness_weight": 2.0,
    },
    {
        "hidden_size": 64,
        "num_layers": 1,
        "dropout": 0.30,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "correctness_weight": 2.5,
    },
    {
        "hidden_size": 64,
        "num_layers": 2,
        "dropout": 0.35,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "correctness_weight": 2.5,
    },
    {
        "hidden_size": 96,
        "num_layers": 2,
        "dropout": 0.40,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "correctness_weight": 3.0,
    },
    {
        "hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.50,
        "lr": 3e-4,
        "weight_decay": 1e-3,
        "correctness_weight": 3.0,
    },
    {
        "hidden_size": 192,
        "num_layers": 2,
        "dropout": 0.45,
        "lr": 2e-4,
        "weight_decay": 1e-3,
        "correctness_weight": 3.0,
    },
    {
        "hidden_size": 256,
        "num_layers": 3,
        "dropout": 0.50,
        "lr": 1e-4,
        "weight_decay": 2e-3,
        "correctness_weight": 3.0,
    },
]

TCN_GRID = [
    # Config 1 — lightweight, extended receptive field baseline
    {
        "channels":   [32, 32, 64, 64],
        "kernel_size": 3,
        "dilations":  [1, 2, 4, 8],        # RF = 1 + 4×(1+2+4+8) = 61  ← just covers most reps
        "dropout": 0.25,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "correctness_weight": 2.0,
    },
    # Config 2 — same RF, wider channels, more correctness pressure
    {
        "channels":   [32, 64, 64, 128],
        "kernel_size": 3,
        "dilations":  [1, 2, 4, 8],        # RF = 61
        "dropout": 0.30,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "correctness_weight": 2.5,
    },
    # Config 3 — full RF coverage, moderate capacity
    {
        "channels":   [32, 64, 64, 64, 64],
        "kernel_size": 3,
        "dilations":  [1, 2, 4, 8, 16],    # RF = 1 + 4×(1+2+4+8+16) = 125 ← covers full 90 frames
        "dropout": 0.30,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "correctness_weight": 2.5,
    },
    # Config 4 — full RF coverage, higher capacity
    {
        "channels":   [32, 64, 64, 128, 128],
        "kernel_size": 3,
        "dilations":  [1, 2, 4, 8, 16],    # RF = 125
        "dropout": 0.35,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "correctness_weight": 3.0,
    },
    # Config 5 — full RF coverage, aggressive regularisation
    {
        "channels":   [32, 64, 128, 128, 128],
        "kernel_size": 3,
        "dilations":  [1, 2, 4, 8, 16],    # RF = 125
        "dropout": 0.40,
        "lr": 3e-4,
        "weight_decay": 1e-3,
        "correctness_weight": 3.0,
    },
    # Config 6 — tiny model, strong regularisation (anti-overfit baseline)
    {
        "channels":   [16, 32, 32, 64],
        "kernel_size": 3,
        "dilations":  [1, 2, 4, 8],        # RF = 61
        "dropout": 0.30,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "correctness_weight": 2.0,
    },
    # Config 7 — tiny model, full RF coverage
    {
        "channels":   [16, 32, 32, 64, 64],
        "kernel_size": 3,
        "dilations":  [1, 2, 4, 8, 16],    # RF = 125
        "dropout": 0.35,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "correctness_weight": 2.5,
    },
    # Config 8 — medium, wider kernel for richer local context per block
    {
        "channels":   [32, 32, 64, 64],
        "kernel_size": 5,
        "dilations":  [1, 2, 4, 8],        # RF = 1 + 8*(1+2+4+8) = 121
        "dropout": 0.35,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "correctness_weight": 2.5,
    },
]


# ============================================================
# 8. Training function
# ============================================================

def train_model(model, model_name, cfg, save_path, seed=42):
    set_seed(seed)

    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"]
    )

    criterion_ex = nn.CrossEntropyLoss(weight=exercise_class_weights)
    criterion_cor = nn.CrossEntropyLoss(weight=correctness_class_weights)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=4
    )

    best_score = -1.0
    best_checkpoint = None
    epochs_no_improve = 0
    early_stop_patience = 10

    history = []

    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        train_loss = 0.0

        for X_batch, y_ex_batch, y_cor_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_ex_batch = y_ex_batch.to(DEVICE)
            y_cor_batch = y_cor_batch.to(DEVICE)

            optimizer.zero_grad()

            logits_ex, logits_cor = model(X_batch)

            loss_ex = criterion_ex(logits_ex, y_ex_batch)
            loss_cor = criterion_cor(logits_cor, y_cor_batch)

            loss = loss_ex + cfg["correctness_weight"] * loss_cor

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        metrics = evaluate_model(
            model,
            val_loader,
            video_ids=VAL_WINDOW_VIDEO_IDS
        )

        score = metrics["score"]
        scheduler.step(score)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            **metrics,
        }
        history.append(row)

        print(
            f"  Epoch {epoch:02d}/{N_EPOCHS} | "
            f"Loss: {train_loss:.4f} | "
            f"Win F1 Ex: {metrics['window_f1_ex_macro']:.4f} | "
            f"Win F1 Cor: {metrics['window_f1_cor_macro']:.4f} | "
            f"Score: {score:.4f}"
            + (" < best" if score > best_score else "")
        )

        if score > best_score:
            best_score = score
            best_checkpoint = {
                "epoch": epoch,
                "model_name": model_name,
                "model_config": cfg,
                "model_state": copy.deepcopy(model.state_dict()),
                "optimizer_state": copy.deepcopy(optimizer.state_dict()),
                "metrics": metrics,
                "score": score,
                "seed": seed,
                "sequence_length": SEQUENCE_LENGTH,
                "feature_cols": feature_cols,
                "train_mean": train_mean.cpu(),
                "train_std": train_std.cpu(),
                "exercise_class_weights": exercise_class_weights.detach().cpu(),
                "correctness_class_weights": correctness_class_weights.detach().cpu(),
            }
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

            if epochs_no_improve >= early_stop_patience:
                print(f"  Early stopping at epoch {epoch}.")
                break

    torch.save(best_checkpoint, save_path)

    print(f"  Best score: {best_score:.4f}")
    print(f"  Saved to: {save_path}\n")

    return {
        "model_name": model_name,
        "config": cfg,
        "best_score": best_score,
        "best_metrics": best_checkpoint["metrics"],
        "checkpoint_path": save_path,
        "history": history,
    }


# ============================================================
# 9. Run grid search
# ============================================================

all_results = []
SEARCH_SEED = 42

print("=" * 60)
print("LSTM GRID SEARCH")
print("=" * 60)

for i, cfg in enumerate(LSTM_GRID, start=1):
    print(f"\n[LSTM {i}/{len(LSTM_GRID)}]")
    print(cfg)

    set_seed(SEARCH_SEED)
    model = build_lstm(cfg)

    result = train_model(
        model=model,
        model_name=f"LSTM_cfg{i}",
        cfg=cfg,
        save_path=MODEL_DIR / f"LSTM_cfg{i}.pt",
        seed=SEARCH_SEED,
    )

    all_results.append(result)


print("=" * 60)
print("TCN GRID SEARCH")
print("=" * 60)

for i, cfg in enumerate(TCN_GRID, start=1):
    print(f"\n[TCN {i}/{len(TCN_GRID)}]")
    print(cfg)

    set_seed(SEARCH_SEED)
    model = build_tcn(cfg)

    result = train_model(
        model=model,
        model_name=f"TCN_cfg{i}",
        cfg=cfg,
        save_path=MODEL_DIR / f"TCN_cfg{i}.pt",
        seed=SEARCH_SEED,
    )

    all_results.append(result)


# ============================================================
# 10. Summary and final checkpoint copies
# ============================================================

print("=" * 60)
print("GRID SEARCH SUMMARY")
print("=" * 60)

all_results_sorted = sorted(
    all_results,
    key=lambda r: r["best_score"],
    reverse=True
)

for rank, result in enumerate(all_results_sorted, start=1):
    metrics = result["best_metrics"]

    print(f"\nRank {rank}: {result['model_name']}")
    print(f"  Score: {result['best_score']:.4f}")
    print(f"  Window F1 Exercise Macro   : {metrics['window_f1_ex_macro']:.4f}")
    print(f"  Window F1 Correctness Macro: {metrics['window_f1_cor_macro']:.4f}")

    if "video_f1_ex_macro" in metrics:
        print(f"  Video F1 Exercise Macro    : {metrics['video_f1_ex_macro']:.4f}")
        print(f"  Video F1 Correctness Macro : {metrics['video_f1_cor_macro']:.4f}")

    print(f"  Config: {result['config']}")


best_lstm = max(
    [r for r in all_results if r["model_name"].startswith("LSTM")],
    key=lambda r: r["best_score"]
)

best_tcn = max(
    [r for r in all_results if r["model_name"].startswith("TCN")],
    key=lambda r: r["best_score"]
)

shutil.copy(best_lstm["checkpoint_path"], MODEL_DIR / "LSTM.pt")
shutil.copy(best_tcn["checkpoint_path"], MODEL_DIR / "TCN.pt")

print("\nBest LSTM:")
print(f"  {best_lstm['model_name']} | Score: {best_lstm['best_score']:.4f}")
print(f"  Saved as: {MODEL_DIR / 'LSTM.pt'}")

print("\nBest TCN:")
print(f"  {best_tcn['model_name']} | Score: {best_tcn['best_score']:.4f}")
print(f"  Saved as: {MODEL_DIR / 'TCN.pt'}")

overall_best = all_results_sorted[0]

print("\nOverall best model:")
print(f"  {overall_best['model_name']} | Score: {overall_best['best_score']:.4f}")
print(f"  Checkpoint: {overall_best['checkpoint_path']}")

print("\nCell 4 complete.")

Video split check passed.
Train videos: 170
Val videos  : 37
Test videos : 37
Train-only feature scaling applied.
Exercise class weights   : [0.9057971 1.1160715 1.0775862 0.9328358]
Correctness class weights: [0.9398496 1.0683761]
Video-level validation enabled.
LSTM GRID SEARCH

[LSTM 1/7]
{'hidden_size': 32, 'num_layers': 1, 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 0.0001, 'correctness_weight': 2.0}
  Epoch 01/50 | Loss: 2.7309 | Win F1 Ex: 0.6114 | Win F1 Cor: 0.3294 | Score: 0.3681 < best
  Epoch 02/50 | Loss: 2.6412 | Win F1 Ex: 0.6129 | Win F1 Cor: 0.3294 | Score: 0.3648
  Epoch 03/50 | Loss: 2.5263 | Win F1 Ex: 0.7798 | Win F1 Cor: 0.4019 | Score: 0.4795 < best
  Epoch 04/50 | Loss: 2.3692 | Win F1 Ex: 0.8562 | Win F1 Cor: 0.4962 | Score: 0.5605 < best
  Epoch 05/50 | Loss: 2.1850 | Win F1 Ex: 0.8622 | Win F1 Cor: 0.4855 | Score: 0.5645 < best
  Epoch 06/50 | Loss: 1.9960 | Win F1 Ex: 0.8622 | Win F1 Cor: 0.5074 | Score: 0.5797 < best
  Epoch 07/50 | Loss: 1.8618 | Win F1 E

c:\Users\USER\Desktop\AI_PROJECT_2\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Epoch 01/50 | Loss: 2.7084 | Win F1 Ex: 0.5665 | Win F1 Cor: 0.3294 | Score: 0.3414 < best
  Epoch 02/50 | Loss: 2.4733 | Win F1 Ex: 0.3640 | Win F1 Cor: 0.4477 | Score: 0.4152 < best
  Epoch 03/50 | Loss: 2.2930 | Win F1 Ex: 0.9499 | Win F1 Cor: 0.3798 | Score: 0.5335 < best
  Epoch 04/50 | Loss: 2.0184 | Win F1 Ex: 0.7085 | Win F1 Cor: 0.5437 | Score: 0.5554 < best
  Epoch 05/50 | Loss: 1.7416 | Win F1 Ex: 0.9349 | Win F1 Cor: 0.4818 | Score: 0.5702 < best
  Epoch 06/50 | Loss: 1.5774 | Win F1 Ex: 0.9339 | Win F1 Cor: 0.4146 | Score: 0.5412
  Epoch 07/50 | Loss: 1.4947 | Win F1 Ex: 0.9677 | Win F1 Cor: 0.4887 | Score: 0.5660
  Epoch 08/50 | Loss: 1.3834 | Win F1 Ex: 0.9514 | Win F1 Cor: 0.3667 | Score: 0.4774
  Epoch 09/50 | Loss: 1.4573 | Win F1 Ex: 0.9516 | Win F1 Cor: 0.5369 | Score: 0.6704 < best
  Epoch 10/50 | Loss: 1.4618 | Win F1 Ex: 0.9514 | Win F1 Cor: 0.4818 | Score: 0.5729
  Epoch 11/50 | Loss: 1.3146 | Win F1 Ex: 0.9514 | Win F1 Cor: 0.4887 | Score: 0.5660
  Epoch 12/5

c:\Users\USER\Desktop\AI_PROJECT_2\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Epoch 01/50 | Loss: 3.1026 | Win F1 Ex: 0.6001 | Win F1 Cor: 0.3294 | Score: 0.3532 < best
  Epoch 02/50 | Loss: 2.8895 | Win F1 Ex: 0.6119 | Win F1 Cor: 0.4477 | Score: 0.4140 < best
  Epoch 03/50 | Loss: 2.7417 | Win F1 Ex: 0.8474 | Win F1 Cor: 0.5263 | Score: 0.5673 < best
  Epoch 04/50 | Loss: 2.4688 | Win F1 Ex: 0.9150 | Win F1 Cor: 0.3214 | Score: 0.4040
  Epoch 05/50 | Loss: 2.1991 | Win F1 Ex: 0.9308 | Win F1 Cor: 0.4912 | Score: 0.5447
  Epoch 06/50 | Loss: 2.1987 | Win F1 Ex: 0.9502 | Win F1 Cor: 0.5043 | Score: 0.5755 < best
  Epoch 07/50 | Loss: 2.0390 | Win F1 Ex: 0.9839 | Win F1 Cor: 0.5426 | Score: 0.6296 < best
  Epoch 08/50 | Loss: 2.0791 | Win F1 Ex: 0.9677 | Win F1 Cor: 0.4962 | Score: 0.5532
  Epoch 09/50 | Loss: 1.9685 | Win F1 Ex: 0.9677 | Win F1 Cor: 0.5592 | Score: 0.6270
  Epoch 10/50 | Loss: 1.8657 | Win F1 Ex: 0.9677 | Win F1 Cor: 0.6110 | Score: 0.7189 < best
  Epoch 11/50 | Loss: 1.7071 | Win F1 Ex: 0.9663 | Win F1 Cor: 0.5788 | Score: 0.6556
  Epoch 12/5

...

  Epoch 01/50 | Loss: 2.7727 | Win F1 Ex: 0.1096 | Win F1 Cor: 0.3294 | Score: 0.2892 < best
  Epoch 02/50 | Loss: 2.7242 | Win F1 Ex: 0.5645 | Win F1 Cor: 0.3609 | Score: 0.3933 < best
  Epoch 03/50 | Loss: 2.5370 | Win F1 Ex: 0.6128 | Win F1 Cor: 0.5627 | Score: 0.5939 < best
  Epoch 04/50 | Loss: 2.2078 | Win F1 Ex: 0.3445 | Win F1 Cor: 0.5778 | Score: 0.5525
  Epoch 05/50 | Loss: 2.1871 | Win F1 Ex: 0.7426 | Win F1 Cor: 0.5050 | Score: 0.5812
  Epoch 06/50 | Loss: 2.1054 | Win F1 Ex: 0.7327 | Win F1 Cor: 0.5257 | Score: 0.5785
  Epoch 07/50 | Loss: 1.9859 | Win F1 Ex: 0.8758 | Win F1 Cor: 0.5565 | Score: 0.6032 < best
  Epoch 08/50 | Loss: 1.8459 | Win F1 Ex: 0.7624 | Win F1 Cor: 0.4858 | Score: 0.5666
  Epoch 09/50 | Loss: 1.8915 | Win F1 Ex: 0.9323 | Win F1 Cor: 0.5263 | Score: 0.5702
  Epoch 10/50 | Loss: 1.6969 | Win F1 Ex: 0.9150 | Win F1 Cor: 0.6130 | Score: 0.6378 < best
  Epoch 11/50 | Loss: 1.6932 | Win F1 Ex: 0.9180 | Win F1 Cor: 0.5168 | Score: 0.6280
  Epoch 12/50 | Los

c:\Users\USER\Desktop\AI_PROJECT_2\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Epoch 01/50 | Loss: 3.1195 | Win F1 Ex: 0.0929 | Win F1 Cor: 0.3372 | Score: 0.3056 < best
  Epoch 02/50 | Loss: 3.0995 | Win F1 Ex: 0.2250 | Win F1 Cor: 0.3372 | Score: 0.3263 < best
  Epoch 03/50 | Loss: 3.0623 | Win F1 Ex: 0.6273 | Win F1 Cor: 0.3372 | Score: 0.3843 < best
  Epoch 04/50 | Loss: 2.9559 | Win F1 Ex: 0.6151 | Win F1 Cor: 0.5369 | Score: 0.5627 < best
  Epoch 05/50 | Loss: 2.7548 | Win F1 Ex: 0.4059 | Win F1 Cor: 0.4534 | Score: 0.4550
  Epoch 06/50 | Loss: 2.5891 | Win F1 Ex: 0.5790 | Win F1 Cor: 0.4534 | Score: 0.4812
  Epoch 07/50 | Loss: 2.5420 | Win F1 Ex: 0.6356 | Win F1 Cor: 0.4722 | Score: 0.5116
  Epoch 08/50 | Loss: 2.4454 | Win F1 Ex: 0.6229 | Win F1 Cor: 0.5086 | Score: 0.5690 < best
  Epoch 09/50 | Loss: 2.3317 | Win F1 Ex: 0.6205 | Win F1 Cor: 0.4586 | Score: 0.5135
  Epoch 10/50 | Loss: 2.2433 | Win F1 Ex: 0.6356 | Win F1 Cor: 0.5257 | Score: 0.5516
  Epoch 11/50 | Loss: 2.1827 | Win F1 Ex: 0.7924 | Win F1 Cor: 0.4722 | Score: 0.5604
  Epoch 12/50 | Los

c:\Users\USER\Desktop\AI_PROJECT_2\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Epoch 01/50 | Loss: 3.0877 | Win F1 Ex: 0.3486 | Win F1 Cor: 0.4384 | Score: 0.4821 < best
  Epoch 02/50 | Loss: 2.9841 | Win F1 Ex: 0.5045 | Win F1 Cor: 0.5908 | Score: 0.6048 < best
  Epoch 03/50 | Loss: 2.7698 | Win F1 Ex: 0.4189 | Win F1 Cor: 0.3667 | Score: 0.3210
  Epoch 04/50 | Loss: 2.6085 | Win F1 Ex: 0.6954 | Win F1 Cor: 0.3294 | Score: 0.3625
  Epoch 05/50 | Loss: 2.6865 | Win F1 Ex: 0.9020 | Win F1 Cor: 0.3197 | Score: 0.4627
  Epoch 06/50 | Loss: 2.5076 | Win F1 Ex: 0.8539 | Win F1 Cor: 0.5565 | Score: 0.6499 < best
  Epoch 07/50 | Loss: 2.3969 | Win F1 Ex: 0.9175 | Win F1 Cor: 0.5086 | Score: 0.5798
  Epoch 08/50 | Loss: 2.1909 | Win F1 Ex: 0.9180 | Win F1 Cor: 0.5257 | Score: 0.6195
  Epoch 09/50 | Loss: 2.1092 | Win F1 Ex: 0.9354 | Win F1 Cor: 0.4316 | Score: 0.5558
  Epoch 10/50 | Loss: 2.0238 | Win F1 Ex: 0.9354 | Win F1 Cor: 0.6130 | Score: 0.7114 < best
  Epoch 11/50 | Loss: 1.8190 | Win F1 Ex: 0.9516 | Win F1 Cor: 0.5188 | Score: 0.6646
  Epoch 12/50 | Loss: 1.90

In [5]:
# ── Architecture assertion ────────────────────────────────────────────
# The Cell 3 sanity check used default (small) configs for the forward-pass test.
# Here we load the actually saved checkpoints and verify their parameter counts
# so that notebook 06 (webcam inference) loads architecturally consistent models.

import torch

_training_keys = {"lr", "weight_decay", "correctness_weight"}

def verify_checkpoint(path, model_builder, label):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    arch_cfg = {k: v for k, v in ckpt["model_config"].items()
                if k not in _training_keys}
    model = model_builder(arch_cfg)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Confirm state dict loads without key mismatches
    model.load_state_dict(ckpt["model_state"])

    print(f"✅ {label}")
    print(f"   Checkpoint epoch : {ckpt['epoch']}")
    print(f"   Architecture     : {arch_cfg}")
    print(f"   Trainable params : {n_params:,}")
    print(f"   Val score        : {ckpt['score']:.4f}")
    return n_params

print("=" * 60)
print("CHECKPOINT ARCHITECTURE VERIFICATION")
print("=" * 60)

lstm_params = verify_checkpoint(MODEL_DIR / "LSTM.pt", build_lstm, "LSTM.pt")
print()
tcn_params  = verify_checkpoint(MODEL_DIR / "TCN.pt",  build_tcn,  "TCN.pt")

print()
print("Note for webcam inference (notebook 06):")
print(f"  LSTM trainable params : {lstm_params:,}")
print(f"  TCN  trainable params : {tcn_params:,}")
print("  Both checkpoints verified. load_state_dict() passed without errors.")


CHECKPOINT ARCHITECTURE VERIFICATION
✅ LSTM.pt
   Checkpoint epoch : 50
   Architecture     : {'hidden_size': 192, 'num_layers': 2, 'dropout': 0.45}
   Trainable params : 1,325,261
   Val score        : 0.9744

✅ TCN.pt
   Checkpoint epoch : 27
   Architecture     : {'channels': [32, 64, 64, 128], 'kernel_size': 3, 'dilations': [1, 2, 4, 8], 'dropout': 0.3}
   Trainable params : 143,788
   Val score        : 0.9718

Note for webcam inference (notebook 06):
  LSTM trainable params : 1,325,261
  TCN  trainable params : 143,788
  Both checkpoints verified. load_state_dict() passed without errors.


c:\Users\USER\Desktop\AI_PROJECT_2\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [6]:
# ============================================================
# Cell 5 - Evaluation & Per-Class Diagnosis
# Matches the optimized Cell 4 checkpoint format
# ============================================================

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

EXERCISE_NAMES = ["pullup", "pushup", "russian_twist", "squat"]
CORRECTNESS_NAMES = ["incorrect", "correct"]


# ============================================================
# 1. Test-set scaling using checkpoint statistics
# ============================================================

def apply_checkpoint_scaling_to_test(checkpoint):
    """
    Applies the same train-only scaling used in Cell 4.
    Safe to run multiple times because it always scales from X_raw.
    """

    if "train_mean" not in checkpoint or "train_std" not in checkpoint:
        print("Warning: checkpoint does not contain scaling statistics.")
        return

    if not hasattr(test_dataset, "X_raw"):
        test_dataset.X_raw = test_dataset.X.clone()

    mean = checkpoint["train_mean"].to(test_dataset.X_raw.device)
    std = checkpoint["train_std"].to(test_dataset.X_raw.device)
    std[std == 0] = 1.0

    test_dataset.X = (test_dataset.X_raw - mean) / std

    print("Test set scaled using checkpoint train_mean/train_std.")


# ============================================================
# 2. Build test window video IDs
# ============================================================

def build_window_video_ids(csv_path, sequence_length):
    df = pd.read_csv(csv_path)
    window_video_ids = []

    for video_id, group in df.groupby("video_id"):
        group = group.sort_values("frame_number").reset_index(drop=True)
        n_frames = len(group)

        win_size = min(n_frames, sequence_length)

        for start in range(0, n_frames - win_size + 1, win_size):
            window_video_ids.append(video_id)

        remainder = n_frames % win_size
        if remainder > 20:
            window_video_ids.append(video_id)

    return np.array(window_video_ids)


def get_test_window_video_ids(checkpoint):
    sequence_length = checkpoint.get("sequence_length", SEQUENCE_LENGTH)
    video_ids = build_window_video_ids(TEST_CSV, sequence_length)

    if len(video_ids) != len(test_dataset):
        print("Warning: video-level test evaluation disabled because window IDs do not match dataset length.")
        return None

    return video_ids


# ============================================================
# 3. Model builders
# ============================================================

def build_lstm_from_config(cfg):
    try:
        return ExerciseLSTM(
            input_size=N_FEATURES,
            n_exercise=N_EXERCISE_CLASSES,
            n_correctness=N_CORRECTNESS_CLASSES,
            hidden_size=cfg["hidden_size"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"],
        )
    except TypeError:
        return ExerciseLSTM(
            input_size=N_FEATURES,
            hidden_size=cfg["hidden_size"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"],
        )


def build_tcn_from_config(cfg):
    try:
        return ExerciseTCN(
            input_size=N_FEATURES,
            channels=cfg["channels"],
            kernel_size=cfg.get("kernel_size", 3),
            dilations=cfg.get("dilations", None),
            dropout=cfg["dropout"],
        )
    except TypeError:
        return ExerciseTCN(
            channels=cfg["channels"],
            dropout=cfg["dropout"],
        )


def build_model_from_checkpoint(checkpoint, model_label):
    cfg = checkpoint["model_config"]

    if model_label.lower() == "lstm":
        return build_lstm_from_config(cfg)

    if model_label.lower() == "tcn":
        return build_tcn_from_config(cfg)

    raise ValueError(f"Unknown model label: {model_label}")


# ============================================================
# 4. Inference helpers
# ============================================================

def run_inference(model, loader):
    model.eval()

    all_ex_probs = []
    all_cor_probs = []
    all_ex_preds = []
    all_cor_preds = []
    all_ex_labels = []
    all_cor_labels = []

    with torch.no_grad():
        for X_batch, y_ex_batch, y_cor_batch in loader:
            X_batch = X_batch.to(DEVICE)

            logits_ex, logits_cor = model(X_batch)

            ex_probs = torch.softmax(logits_ex, dim=1).cpu().numpy()
            cor_probs = torch.softmax(logits_cor, dim=1).cpu().numpy()

            ex_preds = ex_probs.argmax(axis=1)
            cor_preds = cor_probs.argmax(axis=1)

            all_ex_probs.append(ex_probs)
            all_cor_probs.append(cor_probs)

            all_ex_preds.extend(ex_preds)
            all_cor_preds.extend(cor_preds)
            all_ex_labels.extend(y_ex_batch.numpy())
            all_cor_labels.extend(y_cor_batch.numpy())

    return {
        "ex_probs": np.vstack(all_ex_probs),
        "cor_probs": np.vstack(all_cor_probs),
        "y_ex_true": np.array(all_ex_labels),
        "y_ex_pred": np.array(all_ex_preds),
        "y_cor_true": np.array(all_cor_labels),
        "y_cor_pred": np.array(all_cor_preds),
    }


def majority_label(labels):
    labels = np.asarray(labels, dtype=int)
    return int(np.bincount(labels).argmax())


def aggregate_by_video(video_ids, inference_output):
    video_ids = np.asarray(video_ids)

    ex_probs = inference_output["ex_probs"]
    cor_probs = inference_output["cor_probs"]

    y_ex_true = inference_output["y_ex_true"]
    y_cor_true = inference_output["y_cor_true"]

    video_ex_preds = []
    video_ex_labels = []
    video_cor_preds = []
    video_cor_labels = []

    for video_id in np.unique(video_ids):
        idx = np.where(video_ids == video_id)[0]

        ex_pred = int(ex_probs[idx].mean(axis=0).argmax())
        cor_pred = int(cor_probs[idx].mean(axis=0).argmax())

        ex_label = majority_label(y_ex_true[idx])
        cor_label = majority_label(y_cor_true[idx])

        video_ex_preds.append(ex_pred)
        video_ex_labels.append(ex_label)
        video_cor_preds.append(cor_pred)
        video_cor_labels.append(cor_label)

    return {
        "video_y_ex_true": np.array(video_ex_labels),
        "video_y_ex_pred": np.array(video_ex_preds),
        "video_y_cor_true": np.array(video_cor_labels),
        "video_y_cor_pred": np.array(video_cor_preds),
    }


# ============================================================
# 5. Evaluation helper
# ============================================================

def evaluate_checkpoint(checkpoint_path, model_label):
    print("=" * 60)
    print(f"  {model_label}")
    print("=" * 60)

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

    apply_checkpoint_scaling_to_test(checkpoint)

    model = build_model_from_checkpoint(checkpoint, model_label)
    model.load_state_dict(checkpoint["model_state"])
    model.to(DEVICE)

    cfg = checkpoint["model_config"]
    saved_metrics = checkpoint.get("metrics", {})
    saved_score = checkpoint.get("score", None)
    saved_epoch = checkpoint.get("epoch", None)

    print("\nCheckpoint info:")
    print(f"  Saved at epoch: {saved_epoch}")
    print(f"  Config        : {cfg}")

    if saved_score is not None:
        print(f"  Val score     : {saved_score:.4f}")

    if saved_metrics:
        print("\nValidation metrics from Cell 4:")
        for key, value in saved_metrics.items():
            if isinstance(value, (int, float)):
                print(f"  {key}: {value:.4f}")

    test_video_ids = get_test_window_video_ids(checkpoint)

    output = run_inference(model, test_loader)

    y_ex_true = output["y_ex_true"]
    y_ex_pred = output["y_ex_pred"]
    y_cor_true = output["y_cor_true"]
    y_cor_pred = output["y_cor_pred"]

    # ---------------- Window-level metrics ----------------

    test_acc_ex = accuracy_score(y_ex_true, y_ex_pred)
    test_acc_cor = accuracy_score(y_cor_true, y_cor_pred)

    test_f1_ex_macro = f1_score(
        y_ex_true,
        y_ex_pred,
        average="macro",
        zero_division=0
    )

    test_f1_cor_macro = f1_score(
        y_cor_true,
        y_cor_pred,
        average="macro",
        zero_division=0
    )

    test_f1_ex_weighted = f1_score(
        y_ex_true,
        y_ex_pred,
        average="weighted",
        zero_division=0
    )

    test_f1_cor_weighted = f1_score(
        y_cor_true,
        y_cor_pred,
        average="weighted",
        zero_division=0
    )

    test_score_window = 0.15 * test_f1_ex_macro + 0.85 * test_f1_cor_macro

    print("\nWindow-level Test Results:")
    print(f"  Exercise    - Accuracy: {test_acc_ex:.4f} | Macro F1: {test_f1_ex_macro:.4f} | Weighted F1: {test_f1_ex_weighted:.4f}")
    print(f"  Correctness - Accuracy: {test_acc_cor:.4f} | Macro F1: {test_f1_cor_macro:.4f} | Weighted F1: {test_f1_cor_weighted:.4f}")
    print(f"  Window score: {test_score_window:.4f}")

    print("\nExercise Head - Window-level Classification Report:")
    print(classification_report(
        y_ex_true,
        y_ex_pred,
        target_names=EXERCISE_NAMES,
        zero_division=0
    ))

    print("Correctness Head - Window-level Classification Report:")
    print(classification_report(
        y_cor_true,
        y_cor_pred,
        target_names=CORRECTNESS_NAMES,
        zero_division=0
    ))

    # ---------------- Video-level metrics ----------------

    video_output = None

    if test_video_ids is not None:
        video_output = aggregate_by_video(test_video_ids, output)

        video_y_ex_true = video_output["video_y_ex_true"]
        video_y_ex_pred = video_output["video_y_ex_pred"]
        video_y_cor_true = video_output["video_y_cor_true"]
        video_y_cor_pred = video_output["video_y_cor_pred"]

        video_acc_ex = accuracy_score(video_y_ex_true, video_y_ex_pred)
        video_acc_cor = accuracy_score(video_y_cor_true, video_y_cor_pred)

        video_f1_ex_macro = f1_score(
            video_y_ex_true,
            video_y_ex_pred,
            average="macro",
            zero_division=0
        )

        video_f1_cor_macro = f1_score(
            video_y_cor_true,
            video_y_cor_pred,
            average="macro",
            zero_division=0
        )

        video_f1_ex_weighted = f1_score(
            video_y_ex_true,
            video_y_ex_pred,
            average="weighted",
            zero_division=0
        )

        video_f1_cor_weighted = f1_score(
            video_y_cor_true,
            video_y_cor_pred,
            average="weighted",
            zero_division=0
        )

        video_score = 0.15 * video_f1_ex_macro + 0.85 * video_f1_cor_macro

        print("\nVideo-level Test Results:")
        print(f"  Exercise    - Accuracy: {video_acc_ex:.4f} | Macro F1: {video_f1_ex_macro:.4f} | Weighted F1: {video_f1_ex_weighted:.4f}")
        print(f"  Correctness - Accuracy: {video_acc_cor:.4f} | Macro F1: {video_f1_cor_macro:.4f} | Weighted F1: {video_f1_cor_weighted:.4f}")
        print(f"  Video score : {video_score:.4f}")

        print("\nExercise Head - Video-level Classification Report:")
        print(classification_report(
            video_y_ex_true,
            video_y_ex_pred,
            target_names=EXERCISE_NAMES,
            zero_division=0
        ))

        print("Correctness Head - Video-level Classification Report:")
        print(classification_report(
            video_y_cor_true,
            video_y_cor_pred,
            target_names=CORRECTNESS_NAMES,
            zero_division=0
        ))

    # ---------------- Confusion matrices ----------------

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{model_label} - Window-level Test Confusion Matrices", fontsize=13)

    cm_ex = confusion_matrix(y_ex_true, y_ex_pred)
    sns.heatmap(
        cm_ex,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=EXERCISE_NAMES,
        yticklabels=EXERCISE_NAMES,
        ax=axes[0]
    )
    axes[0].set_title("Exercise Head")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")
    axes[0].tick_params(axis="x", rotation=30)

    cm_cor = confusion_matrix(y_cor_true, y_cor_pred)
    sns.heatmap(
        cm_cor,
        annot=True,
        fmt="d",
        cmap="Oranges",
        xticklabels=CORRECTNESS_NAMES,
        yticklabels=CORRECTNESS_NAMES,
        ax=axes[1]
    )
    axes[1].set_title("Correctness Head")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Actual")

    plt.tight_layout()

    plot_path = FIGURES_DIR / f"{model_label}_window_confusion_matrices.png"
    plt.savefig(plot_path, dpi=120, bbox_inches="tight")
    plt.show()

    print(f"\nWindow-level confusion matrices saved to: {plot_path}")

    if video_output is not None:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f"{model_label} - Video-level Test Confusion Matrices", fontsize=13)

        cm_ex_video = confusion_matrix(
            video_output["video_y_ex_true"],
            video_output["video_y_ex_pred"]
        )

        sns.heatmap(
            cm_ex_video,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=EXERCISE_NAMES,
            yticklabels=EXERCISE_NAMES,
            ax=axes[0]
        )
        axes[0].set_title("Exercise Head")
        axes[0].set_xlabel("Predicted")
        axes[0].set_ylabel("Actual")
        axes[0].tick_params(axis="x", rotation=30)

        cm_cor_video = confusion_matrix(
            video_output["video_y_cor_true"],
            video_output["video_y_cor_pred"]
        )

        sns.heatmap(
            cm_cor_video,
            annot=True,
            fmt="d",
            cmap="Oranges",
            xticklabels=CORRECTNESS_NAMES,
            yticklabels=CORRECTNESS_NAMES,
            ax=axes[1]
        )
        axes[1].set_title("Correctness Head")
        axes[1].set_xlabel("Predicted")
        axes[1].set_ylabel("Actual")

        plt.tight_layout()

        video_plot_path = FIGURES_DIR / f"{model_label}_video_confusion_matrices.png"
        plt.savefig(video_plot_path, dpi=120, bbox_inches="tight")
        plt.show()

        print(f"Video-level confusion matrices saved to: {video_plot_path}")

    result = {
        "model_label": model_label,
        "checkpoint": checkpoint,
        "config": cfg,
        "window_test_acc_ex": test_acc_ex,
        "window_test_acc_cor": test_acc_cor,
        "window_test_f1_ex_macro": test_f1_ex_macro,
        "window_test_f1_cor_macro": test_f1_cor_macro,
        "window_test_f1_ex_weighted": test_f1_ex_weighted,
        "window_test_f1_cor_weighted": test_f1_cor_weighted,
        "window_score": test_score_window,
        "y_ex_true": y_ex_true,
        "y_ex_pred": y_ex_pred,
        "y_cor_true": y_cor_true,
        "y_cor_pred": y_cor_pred,
    }

    if video_output is not None:
        result.update(video_output)
        result["video_score"] = video_score
        result["video_test_f1_ex_macro"] = video_f1_ex_macro
        result["video_test_f1_cor_macro"] = video_f1_cor_macro
        result["video_test_f1_ex_weighted"] = video_f1_ex_weighted
        result["video_test_f1_cor_weighted"] = video_f1_cor_weighted
        result["video_test_acc_ex"] = video_acc_ex
        result["video_test_acc_cor"] = video_acc_cor

    return result


# ============================================================
# 6. Evaluate final LSTM and TCN checkpoints
# ============================================================

lstm_eval = evaluate_checkpoint(
    checkpoint_path=MODEL_DIR / "LSTM.pt",
    model_label="LSTM"
)

tcn_eval = evaluate_checkpoint(
    checkpoint_path=MODEL_DIR / "TCN.pt",
    model_label="TCN"
)


# ============================================================
# 7. Final comparison
# ============================================================

print("\n" + "=" * 60)
print("FINAL TEST COMPARISON")
print("=" * 60)

for result in [lstm_eval, tcn_eval]:
    print(f"\n{result['model_label']}")

    print(f"  Window F1 Exercise Macro   : {result['window_test_f1_ex_macro']:.4f}")
    print(f"  Window F1 Correctness Macro: {result['window_test_f1_cor_macro']:.4f}")
    print(f"  Window Score              : {result['window_score']:.4f}")

    if "video_score" in result:
        print(f"  Video F1 Exercise Macro    : {result['video_test_f1_ex_macro']:.4f}")
        print(f"  Video F1 Correctness Macro : {result['video_test_f1_cor_macro']:.4f}")
        print(f"  Video Score               : {result['video_score']:.4f}")

print("\nCell 5 complete - evaluation and diagnosis done.")

  LSTM
Test set scaled using checkpoint train_mean/train_std.

Checkpoint info:
  Saved at epoch: 50
  Config        : {'hidden_size': 192, 'num_layers': 2, 'dropout': 0.45, 'lr': 0.0002, 'weight_decay': 0.001, 'correctness_weight': 3.0}
  Val score     : 0.9744

Validation metrics from Cell 4:
  window_f1_ex_macro: 0.9660
  window_f1_cor_macro: 0.9118
  window_f1_ex_weighted: 0.9644
  window_f1_cor_weighted: 0.9120
  video_f1_ex_macro: 0.9828
  video_f1_cor_macro: 0.9729
  video_f1_ex_weighted: 0.9730
  video_f1_cor_weighted: 0.9730
  score: 0.9744

Window-level Test Results:
  Exercise    - Accuracy: 0.9821 | Macro F1: 0.9832 | Weighted F1: 0.9821
  Correctness - Accuracy: 0.9286 | Macro F1: 0.9286 | Weighted F1: 0.9286
  Window score: 0.9368

Exercise Head - Window-level Classification Report:
               precision    recall  f1-score   support

       pullup       0.94      1.00      0.97        16
       pushup       1.00      1.00      1.00        14
russian_twist       1.00  

C:\Users\USER\AppData\Local\Temp\ipykernel_22752\1146597453.py:426: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Window-level confusion matrices saved to: ..\data\figures\07_deep_learning\LSTM_window_confusion_matrices.png


C:\Users\USER\AppData\Local\Temp\ipykernel_22752\1146597453.py:475: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
c:\Users\USER\Desktop\AI_PROJECT_2\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Video-level confusion matrices saved to: ..\data\figures\07_deep_learning\LSTM_video_confusion_matrices.png
  TCN
Test set scaled using checkpoint train_mean/train_std.

Checkpoint info:
  Saved at epoch: 27
  Config        : {'channels': [32, 64, 64, 128], 'kernel_size': 3, 'dilations': [1, 2, 4, 8], 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.0001, 'correctness_weight': 2.5}
  Val score     : 0.9718

Validation metrics from Cell 4:
  window_f1_ex_macro: 0.9493
  window_f1_cor_macro: 0.8586
  window_f1_ex_weighted: 0.9461
  window_f1_cor_weighted: 0.8588
  video_f1_ex_macro: 0.9655
  video_f1_cor_macro: 0.9729
  video_f1_ex_weighted: 0.9458
  video_f1_cor_weighted: 0.9730
  score: 0.9718

Window-level Test Results:
  Exercise    - Accuracy: 0.9821 | Macro F1: 0.9833 | Weighted F1: 0.9822
  Correctness - Accuracy: 0.8571 | Macro F1: 0.8542 | Weighted F1: 0.8557
  Window score: 0.8735

Exercise Head - Window-level Classification Report:
               precision    recall  f1-score   

C:\Users\USER\AppData\Local\Temp\ipykernel_22752\1146597453.py:426: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Window-level confusion matrices saved to: ..\data\figures\07_deep_learning\TCN_window_confusion_matrices.png
Video-level confusion matrices saved to: ..\data\figures\07_deep_learning\TCN_video_confusion_matrices.png

FINAL TEST COMPARISON

LSTM
  Window F1 Exercise Macro   : 0.9832
  Window F1 Correctness Macro: 0.9286
  Window Score              : 0.9368
  Video F1 Exercise Macro    : 0.9827
  Video F1 Correctness Macro : 0.9456
  Video Score               : 0.9512

TCN
  Window F1 Exercise Macro   : 0.9833
  Window F1 Correctness Macro: 0.8542
  Window Score              : 0.8735
  Video F1 Exercise Macro    : 0.9828
  Video F1 Correctness Macro : 0.8899
  Video Score               : 0.9038

Cell 5 complete - evaluation and diagnosis done.


C:\Users\USER\AppData\Local\Temp\ipykernel_22752\1146597453.py:475: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ============================================================
# Cell 6 - Model Comparison Summary
# Matches optimized Cell 4 and updated Cell 5
# ============================================================

from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# 1. Utilities
# ============================================================

try:
    count_parameters
except NameError:
    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)


def get_param_count(eval_result, model_label):
    cfg = eval_result["config"]

    if model_label == "LSTM":
        model = build_lstm_from_config(cfg)
    elif model_label == "TCN":
        model = build_tcn_from_config(cfg)
    else:
        raise ValueError(f"Unknown model label: {model_label}")

    return count_parameters(model)


def per_class_f1(y_true, y_pred, class_names):
    scores = f1_score(
        y_true,
        y_pred,
        labels=list(range(len(class_names))),
        average=None,
        zero_division=0
    )

    return {
        name: round(float(score), 4)
        for name, score in zip(class_names, scores)
    }


def get_eval_level(lstm_eval, tcn_eval):
    if "video_score" in lstm_eval and "video_score" in tcn_eval:
        return "video"
    return "window"


def get_metric(eval_result, level, metric_name):
    if level == "video":
        key = f"video_test_{metric_name}"
    else:
        key = f"window_test_{metric_name}"

    return eval_result[key]


def get_score(eval_result, level):
    if level == "video":
        return eval_result["video_score"]

    return eval_result["window_score"]


def get_labels_and_preds(eval_result, level):
    if level == "video":
        return {
            "y_ex_true": eval_result["video_y_ex_true"],
            "y_ex_pred": eval_result["video_y_ex_pred"],
            "y_cor_true": eval_result["video_y_cor_true"],
            "y_cor_pred": eval_result["video_y_cor_pred"],
        }

    return {
        "y_ex_true": eval_result["y_ex_true"],
        "y_ex_pred": eval_result["y_ex_pred"],
        "y_cor_true": eval_result["y_cor_true"],
        "y_cor_pred": eval_result["y_cor_pred"],
    }


def get_main_config_text(eval_result, model_label):
    cfg = eval_result["config"]

    if model_label == "LSTM":
        return f"hidden={cfg.get('hidden_size')}, layers={cfg.get('num_layers')}"

    if model_label == "TCN":
        return f"channels={cfg.get('channels')}"

    return str(cfg)


# ============================================================
# 2. Choose comparison level
# ============================================================

EVAL_LEVEL = get_eval_level(lstm_eval, tcn_eval)

print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print(f"\nComparison level used: {EVAL_LEVEL.upper()}")

if EVAL_LEVEL == "video":
    print("Using video-level results because they are more realistic.")
else:
    print("Using window-level results because video-level results were unavailable.")


# ============================================================
# 3. Parameter counts
# ============================================================

lstm_params = get_param_count(lstm_eval, "LSTM")
tcn_params = get_param_count(tcn_eval, "TCN")


# ============================================================
# 4. Per-class F1 breakdown
# ============================================================

lstm_data = get_labels_and_preds(lstm_eval, EVAL_LEVEL)
tcn_data = get_labels_and_preds(tcn_eval, EVAL_LEVEL)

lstm_ex_perclass = per_class_f1(
    lstm_data["y_ex_true"],
    lstm_data["y_ex_pred"],
    EXERCISE_NAMES
)

lstm_cor_perclass = per_class_f1(
    lstm_data["y_cor_true"],
    lstm_data["y_cor_pred"],
    CORRECTNESS_NAMES
)

tcn_ex_perclass = per_class_f1(
    tcn_data["y_ex_true"],
    tcn_data["y_ex_pred"],
    EXERCISE_NAMES
)

tcn_cor_perclass = per_class_f1(
    tcn_data["y_cor_true"],
    tcn_data["y_cor_pred"],
    CORRECTNESS_NAMES
)


# ============================================================
# 5. Summary table
# ============================================================

lstm_cfg = lstm_eval["config"]
tcn_cfg = tcn_eval["config"]

lstm_score = get_score(lstm_eval, EVAL_LEVEL)
tcn_score = get_score(tcn_eval, EVAL_LEVEL)

lstm_acc_ex = get_metric(lstm_eval, EVAL_LEVEL, "acc_ex")
lstm_acc_cor = get_metric(lstm_eval, EVAL_LEVEL, "acc_cor")
tcn_acc_ex = get_metric(tcn_eval, EVAL_LEVEL, "acc_ex")
tcn_acc_cor = get_metric(tcn_eval, EVAL_LEVEL, "acc_cor")

lstm_f1_ex_macro = get_metric(lstm_eval, EVAL_LEVEL, "f1_ex_macro")
lstm_f1_cor_macro = get_metric(lstm_eval, EVAL_LEVEL, "f1_cor_macro")
tcn_f1_ex_macro = get_metric(tcn_eval, EVAL_LEVEL, "f1_ex_macro")
tcn_f1_cor_macro = get_metric(tcn_eval, EVAL_LEVEL, "f1_cor_macro")

lstm_f1_ex_weighted = get_metric(lstm_eval, EVAL_LEVEL, "f1_ex_weighted")
lstm_f1_cor_weighted = get_metric(lstm_eval, EVAL_LEVEL, "f1_cor_weighted")
tcn_f1_ex_weighted = get_metric(tcn_eval, EVAL_LEVEL, "f1_ex_weighted")
tcn_f1_cor_weighted = get_metric(tcn_eval, EVAL_LEVEL, "f1_cor_weighted")

print(f"\n{'Metric':<35} {'LSTM':>15} {'TCN':>15}")
print("-" * 70)

print(f"{'Trainable Parameters':<35} {lstm_params:>15,} {tcn_params:>15,}")

print(f"{'Main Config':<35} "
      f"{get_main_config_text(lstm_eval, 'LSTM'):>15} "
      f"{get_main_config_text(tcn_eval, 'TCN'):>15}")

print(f"{'Dropout':<35} "
      f"{lstm_cfg.get('dropout'):>15} "
      f"{tcn_cfg.get('dropout'):>15}")

print(f"{'Learning Rate':<35} "
      f"{lstm_cfg.get('lr'):>15} "
      f"{tcn_cfg.get('lr'):>15}")

print(f"{'Weight Decay':<35} "
      f"{lstm_cfg.get('weight_decay'):>15} "
      f"{tcn_cfg.get('weight_decay'):>15}")

print(f"{'Correctness Loss Weight':<35} "
      f"{lstm_cfg.get('correctness_weight'):>15} "
      f"{tcn_cfg.get('correctness_weight'):>15}")

print()
print(f"{'Accuracy - Exercise':<35} {lstm_acc_ex:>15.4f} {tcn_acc_ex:>15.4f}")
print(f"{'Accuracy - Correctness':<35} {lstm_acc_cor:>15.4f} {tcn_acc_cor:>15.4f}")

print()
print(f"{'Macro F1 - Exercise':<35} {lstm_f1_ex_macro:>15.4f} {tcn_f1_ex_macro:>15.4f}")
print(f"{'Macro F1 - Correctness':<35} {lstm_f1_cor_macro:>15.4f} {tcn_f1_cor_macro:>15.4f}")

print()
print(f"{'Weighted F1 - Exercise':<35} {lstm_f1_ex_weighted:>15.4f} {tcn_f1_ex_weighted:>15.4f}")
print(f"{'Weighted F1 - Correctness':<35} {lstm_f1_cor_weighted:>15.4f} {tcn_f1_cor_weighted:>15.4f}")

print()
print(f"{'Final Score':<35} {lstm_score:>15.4f} {tcn_score:>15.4f}")


# ============================================================
# 6. Per-class results
# ============================================================

print()
print("Per-Class F1 - Exercise Head:")
print(f"  {'Class':<18} {'LSTM':>10} {'TCN':>10}")
print("  " + "-" * 40)

for cls in EXERCISE_NAMES:
    print(f"  {cls:<18} {lstm_ex_perclass[cls]:>10.4f} {tcn_ex_perclass[cls]:>10.4f}")

print()
print("Per-Class F1 - Correctness Head:")
print(f"  {'Class':<18} {'LSTM':>10} {'TCN':>10}")
print("  " + "-" * 40)

for cls in CORRECTNESS_NAMES:
    print(f"  {cls:<18} {lstm_cor_perclass[cls]:>10.4f} {tcn_cor_perclass[cls]:>10.4f}")


# ============================================================
# 7. Bar chart comparison
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f"LSTM vs TCN - {EVAL_LEVEL.capitalize()}-Level Test Comparison", fontsize=13)

width = 0.35

# Plot 1 - Exercise per-class F1
x = np.arange(len(EXERCISE_NAMES))

lstm_ex_vals = [lstm_ex_perclass[c] for c in EXERCISE_NAMES]
tcn_ex_vals = [tcn_ex_perclass[c] for c in EXERCISE_NAMES]

axes[0].bar(x - width / 2, lstm_ex_vals, width, label="LSTM")
axes[0].bar(x + width / 2, tcn_ex_vals, width, label="TCN")
axes[0].set_title("Exercise Head - Per-Class F1")
axes[0].set_xticks(x)
axes[0].set_xticklabels(EXERCISE_NAMES, rotation=25, ha="right")
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("F1 Score")
axes[0].legend()
axes[0].axhline(0.8, linestyle="--", linewidth=0.8)

# Plot 2 - Correctness per-class F1
x2 = np.arange(len(CORRECTNESS_NAMES))

lstm_cor_vals = [lstm_cor_perclass[c] for c in CORRECTNESS_NAMES]
tcn_cor_vals = [tcn_cor_perclass[c] for c in CORRECTNESS_NAMES]

axes[1].bar(x2 - width / 2, lstm_cor_vals, width, label="LSTM")
axes[1].bar(x2 + width / 2, tcn_cor_vals, width, label="TCN")
axes[1].set_title("Correctness Head - Per-Class F1")
axes[1].set_xticks(x2)
axes[1].set_xticklabels(CORRECTNESS_NAMES)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("F1 Score")
axes[1].legend()
axes[1].axhline(0.8, linestyle="--", linewidth=0.8)

# Plot 3 - Final score and parameter count
ax3 = axes[2]
ax3b = ax3.twinx()

models = ["LSTM", "TCN"]
scores = [lstm_score, tcn_score]
params = [lstm_params, tcn_params]

bars = ax3.bar(models, scores, width=0.4)
ax3.set_ylim(0, 1.05)
ax3.set_ylabel("Final Score")
ax3.set_title("Final Score and Parameter Count")
ax3.axhline(0.8, linestyle="--", linewidth=0.8)

ax3b.plot(models, params, linestyle="--", marker="o", label="Params")
ax3b.set_ylabel("Trainable Parameters")

for bar, val in zip(bars, scores):
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.tight_layout()

plot_path = FIGURES_DIR / f"model_comparison_{EVAL_LEVEL}.png"
plt.savefig(plot_path, dpi=120, bbox_inches="tight")
plt.show()

print(f"\nComparison chart saved to: {plot_path}")

# ============================================================
# 8. Side-by-side confusion matrices (LSTM vs TCN)
# ============================================================

import seaborn as sns
from sklearn.metrics import confusion_matrix as sk_confusion_matrix

lstm_preds = get_labels_and_preds(lstm_eval, EVAL_LEVEL)
tcn_preds  = get_labels_and_preds(tcn_eval,  EVAL_LEVEL)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f"LSTM vs TCN — {EVAL_LEVEL.capitalize()}-Level Confusion Matrices",
    fontsize=14
)

for col, (label, preds) in enumerate([("LSTM", lstm_preds), ("TCN", tcn_preds)]):
    # Exercise head (row 0)
    cm_ex = sk_confusion_matrix(preds["y_ex_true"], preds["y_ex_pred"])
    sns.heatmap(
        cm_ex,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=EXERCISE_NAMES,
        yticklabels=EXERCISE_NAMES,
        ax=axes[0, col],
    )
    axes[0, col].set_title(f"{label} — Exercise Head")
    axes[0, col].set_xlabel("Predicted")
    axes[0, col].set_ylabel("Actual")
    axes[0, col].tick_params(axis="x", rotation=30)

    # Correctness head (row 1)
    cm_cor = sk_confusion_matrix(preds["y_cor_true"], preds["y_cor_pred"])
    sns.heatmap(
        cm_cor,
        annot=True,
        fmt="d",
        cmap="Oranges",
        xticklabels=CORRECTNESS_NAMES,
        yticklabels=CORRECTNESS_NAMES,
        ax=axes[1, col],
    )
    axes[1, col].set_title(f"{label} — Correctness Head")
    axes[1, col].set_xlabel("Predicted")
    axes[1, col].set_ylabel("Actual")

plt.tight_layout()
cm_plot_path = FIGURES_DIR / f"model_comparison_confusion_matrices_{EVAL_LEVEL}.png"
plt.savefig(cm_plot_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Confusion matrix comparison saved to: {cm_plot_path}")


# ============================================================
# 9. Recommendation
# ============================================================

print()
print("=" * 60)
print("RECOMMENDATION")
print("=" * 60)

if lstm_score > tcn_score:
    winner = "LSTM"
    winner_eval = lstm_eval
    winner_params = lstm_params
    winner_score = lstm_score
    winner_ex_f1 = lstm_f1_ex_macro
    winner_cor_f1 = lstm_f1_cor_macro

elif tcn_score > lstm_score:
    winner = "TCN"
    winner_eval = tcn_eval
    winner_params = tcn_params
    winner_score = tcn_score
    winner_ex_f1 = tcn_f1_ex_macro
    winner_cor_f1 = tcn_f1_cor_macro

else:
    if lstm_f1_cor_macro >= tcn_f1_cor_macro:
        winner = "LSTM"
        winner_eval = lstm_eval
        winner_params = lstm_params
        winner_score = lstm_score
        winner_ex_f1 = lstm_f1_ex_macro
        winner_cor_f1 = lstm_f1_cor_macro
    else:
        winner = "TCN"
        winner_eval = tcn_eval
        winner_params = tcn_params
        winner_score = tcn_score
        winner_ex_f1 = tcn_f1_ex_macro
        winner_cor_f1 = tcn_f1_cor_macro

print(f"""
Recommended model for inference notebooks: {winner}

Rationale:
  Evaluation level : {EVAL_LEVEL}
  Final score      : {winner_score:.4f}
  Exercise Macro F1: {winner_ex_f1:.4f}
  Correctness F1   : {winner_cor_f1:.4f}
  Parameters       : {winner_params:,}

The score uses the same logic as Cell 4 and Cell 5:
  0.15 * exercise_macro_f1 + 0.85 * correctness_macro_f1

This gives more importance to correctness because exercise type is easier,
while correctness is the more important output for form analysis.

Next steps:
  1. Build 08_deep_learning_inference.ipynb for video-file inference.
  2. Build 09_deep_learning_webcam.ipynb for live webcam inference.
  3. Use models/{winner}.pt as the primary model.
""")

RECOMMENDED_MODEL = winner
RECOMMENDED_MODEL_PATH = MODEL_DIR / f"{winner}.pt"

print(f"RECOMMENDED_MODEL = {RECOMMENDED_MODEL}")
print(f"RECOMMENDED_MODEL_PATH = {RECOMMENDED_MODEL_PATH}")

print("\nCell 6 complete - comparison and recommendation done.")
print("07_deep_learning_temporal.ipynb fully complete.")

c:\Users\USER\Desktop\AI_PROJECT_2\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


MODEL COMPARISON SUMMARY

Comparison level used: VIDEO
Using video-level results because they are more realistic.

Metric                                         LSTM             TCN
----------------------------------------------------------------------
Trainable Parameters                      1,325,261         143,788
Main Config                         hidden=192, layers=2 channels=[32, 64, 64, 128]
Dropout                                        0.45             0.3
Learning Rate                                0.0002           0.001
Weight Decay                                  0.001          0.0001
Correctness Loss Weight                         3.0             2.5

Accuracy - Exercise                          0.9730          0.9730
Accuracy - Correctness                       0.9459          0.8919

Macro F1 - Exercise                          0.9827          0.9828
Macro F1 - Correctness                       0.9456          0.8899

Weighted F1 - Exercise                       0.

C:\Users\USER\AppData\Local\Temp\ipykernel_22752\588067845.py:316: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Comparison chart saved to: ..\data\figures\07_deep_learning\model_comparison_video.png
Confusion matrix comparison saved to: ..\data\figures\07_deep_learning\model_comparison_confusion_matrices_video.png

RECOMMENDATION

Recommended model for inference notebooks: LSTM

Rationale:
  Evaluation level : video
  Final score      : 0.9512
  Exercise Macro F1: 0.9827
  Correctness F1   : 0.9456
  Parameters       : 1,325,261

The score uses the same logic as Cell 4 and Cell 5:
  0.15 * exercise_macro_f1 + 0.85 * correctness_macro_f1

This gives more importance to correctness because exercise type is easier,
while correctness is the more important output for form analysis.

Next steps:
  1. Build 08_deep_learning_inference.ipynb for video-file inference.
  2. Build 09_deep_learning_webcam.ipynb for live webcam inference.
  3. Use models/LSTM.pt as the primary model.

RECOMMENDED_MODEL = LSTM
RECOMMENDED_MODEL_PATH = ..\data\models\LSTM.pt

Cell 6 complete - comparison and recommendation done

C:\Users\USER\AppData\Local\Temp\ipykernel_22752\588067845.py:371: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
